# 02 · Bronze ingestion (REQ-01)
**Purpose:** Use Auto Loader to stream new taxi CSV files from the landing folder into the Bronze Delta table, exactly as they arrived.
**Adds:** `source_file` (which file each row came from) and `ingested_at` (when it was loaded).
**Fault tolerance:** a checkpoint remembers which files are done, so files are never loaded twice.

In [0]:
%run ./00_config

# 00 · Config
All project names and paths in one place. Every other notebook loads this with `%run ./00_config`.

✅ Config loaded. Landing folder: /Volumes/workspace/nyc_taxi/pipeline_files/landing


In [0]:
from pyspark.sql import functions as F

# Tell Auto Loader to watch the landing folder for CSV files
raw_stream = (
    spark.readStream
        .format("cloudFiles")                                          # use Auto Loader
        .option("cloudFiles.format", "csv")                            # the files are CSV
        .option("header", "true")                                      # first line = column names
        .option("cloudFiles.schemaLocation", f"{SCHEMA_BASE}/bronze")  # where to remember the columns
        .load(LANDING_PATH)                                            # the folder to watch
)

raw_stream.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- congestion_surcharge: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
# Add 2 stamps to every row
bronze_stream = (
    raw_stream
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withColumn("ingested_at", F.current_timestamp())
)

print("✅ Stamps added")

✅ Stamps added


In [0]:
# Save the stamped letters into the Bronze table
query = (
    bronze_stream.writeStream
        .format("delta")
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/bronze")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()
print("✅ Bronze finished")

✅ Bronze finished


In [0]:
bronze = spark.table(BRONZE_TABLE)

# 1) How many rows are in Bronze?
print("Bronze rows:", bronze.count())

# 2) How many rows came from each file?
display(bronze.groupBy("source_file").count().orderBy("source_file"))

Bronze rows: 50769555


source_file,count
yellow_tripdata_2019-01.csv.gz,7667792
yellow_tripdata_2019-02.csv.gz,7019375
yellow_tripdata_2019-03.csv.gz,7832545
yellow_tripdata_2019-04.csv.gz,7433139
yellow_tripdata_2019-05.csv.gz,7565261
yellow_tripdata_2019-06.csv.gz,6941024
yellow_tripdata_2019-07.csv.gz,6310419


### ✅ Re-run test (no duplicates)
- First run loaded **22,519,712** rows from 3 files (Jan: 7,667,792 · Feb: 7,019,375 · Mar: 7,832,545).
- Running the ingestion again with no new files added **0 rows**. The total stayed at 22,519,712.
- **Why:** the checkpoint (`_checkpoints/bronze`) records which files were already processed, so Auto Loader only picks up new files. Because the checkpoint is stored in the volume, it survives restarts. That's what makes the pipeline fault-tolerant.